In [ ]:
import jax
import jax.numpy as jnp
!pip install -U flax
import flax
import flax.linen as nn
import optax
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.9/533.9 kB 7.1 MB/s  0:00:00
  Attempting uninstall: flax
    Found existing installation: flax 0.11.2
    Uninstalling flax-0.11.2:
      Successfully uninstalled flax-0.11.2


In [ ]:
import jax, flax
print("jax:", jax.__version__, "flax:", flax.__version__)

jax: 0.11.1 flax: 0.12.12


## MLP projector (input_dim -> final_dim through projection_dim)

In [ ]:
# Project representations from "initial dimension" to "z_dim" through a single layer "projector_dim"
class Projector(nn.Module):
    projector_dim: int
    z_dim: int

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(self.projector_dim)(x)
        x = nn.silu(x)

        x = nn.Dense(self.projector_dim)(x)
        x = nn.silu(x)

        x = nn.Dense(self.z_dim)(x)

        return x

In [ ]:
# Testing out projector class : project frmo dim 384 to 768 through 1024
model = Projector(
    projector_dim=1024,
    z_dim=768
)

x = jnp.ones((2, 257, 384)) # [B, num_patches, emb_dim]

variables = model.init(jax.random.PRNGKey(0), x)
output = model.apply(variables, x)
print(output.shape)


(2, 257, 768)


## Adaptive normalisation (scaling and shifting)

In [ ]:
def modulate(x, shift, scale):
  """
  x:     [B, num_patches, D]
  scale: [B, D]
  shift: [B, D]
  """
  return x * (1.0 + scale[:, None, :]) + shift[:, None, :]

## Timestep embedding and Label embedding

In [ ]:
# Embed timestep using fourier positional emcoding scheme and project that into the required "hidden_size"

class TimestepEmbedder(nn.Module):
    hidden_size: int # Final dim of encoded timestep
    frequency_embedding_size: int = 256 # fourier encoding embedding dimension (number of frequencies * 2)

    @staticmethod
    def positional_embedding(t, dim, max_period=10000):
        """
        t : [B, 1]
        dim : int
        max_period : int
        """
        half = dim // 2

        freqs = jnp.exp(
            -jnp.log(max_period) *
            jnp.arange(half, dtype=jnp.float32) / half
        ) # freq = exp(-log(10000) * i/num_frequencies) for all i from 0 to num_frequencies - 1 [num_frequencies]

        args = t[:, None].astype(jnp.float32) * freqs[None, :] # t[:, None] : [B, 1] freq[None, :] : [1, num_frequencies] args : [B, num_frequencies]

        embedding = jnp.concatenate(
            [jnp.cos(args), jnp.sin(args)],
            axis=-1
        ) # [cos(tw0), cos(tw1) .. cos(tw127), sin(tw0), sin(tw1), ... sin(tw127)] [B, frequency_embedding_size]

        if dim % 2:
            embedding = jnp.concatenate(
                [embedding, jnp.zeros_like(embedding[:, :1])],
                axis=-1
            )

        return embedding

    @nn.compact
    def __call__(self, t):

        t_freq = self.positional_embedding(
            t, self.frequency_embedding_size
        )
        # project from dim: frequency_embedding_size to hidden_size
        x = nn.Dense(self.hidden_size)(t_freq)
        x = nn.silu(x)
        x = nn.Dense(self.hidden_size)(x)

        return x

In [ ]:
class LabelEmbedder(nn.Module):
    """
    Converts integer class labels into learnable embedding vectors.
    Supports label dropout for classifier-free guidance (CFG).
    """

    num_classes: int
    hidden_size: int
    dropout_prob: float

    @nn.compact
    def __call__(self, labels, train, force_drop_ids=None):

        # Learnable embedding lookup table:
        # [num_classes + 1, hidden_size]
        # Extra row represents the null/unconditional class.
        embedding_table = nn.Embed(
            num_embeddings=self.num_classes +
                           int(self.dropout_prob > 0),
            features=self.hidden_size
        ) # nn.Embed essentially creates a lookup table for each class to a corresponding encoded vector of dim : hidden_size (learnable!)

        # Randomly replace some labels with the null class during training.
        if train and self.dropout_prob > 0:

            # [B] Boolean mask: True means drop the class label.
            drop_ids = jax.random.bernoulli(
                self.make_rng("dropout"),
                self.dropout_prob,
                labels.shape
            )

            # Replace dropped labels with the null-class index.
            # labels: [B]
            labels = jnp.where(
                drop_ids,
                self.num_classes,
                labels
            )

        # Optionally force specific samples to use the null class (CFG).
        if force_drop_ids is not None:

            # force_drop_ids: [B]
            labels = jnp.where(
                force_drop_ids == 1,
                self.num_classes,
                labels
            )

        # Lookup corresponding embedding vectors.
        # labels: [B]
        # Output: [B, hidden_size]
        return embedding_table(labels)


## SiT Block

In [ ]:
class SiTBlock(nn.Module):
    """
    AdaLN-Zero Transformer block:
    x' = x + gate_msa * Attention(AdaLN(x))
    x'' = x' + gate_mlp * MLP(AdaLN(x'))

    The conditioning vector c = t_embed + y_embed combines the timestep
    embedding and class-label embedding. An MLP generates six vectors
    (shift, scale, gate for attention and MLP), shared across all tokens.
    """

    hidden_size: int
    num_heads: int
    mlp_ratio: float = 4.0
    qk_norm: bool = False

    @nn.compact
    def __call__(self, x, c):

        # x: [B, T, D] — token sequence (patch tokens + CLS token)
        # c: [B, D]    — conditioning = timestep embedding + class embedding

        # Attention normalization: LayerNorm without learnable affine parameters
        h = nn.LayerNorm(
            epsilon=1e-6,
            use_scale=False,
            use_bias=False
        )(x)  # [B, T, D]

        # Generate six conditioning vectors from c through SiLU + Dense.
        # modulation: [B, 6D]
        modulation = nn.Dense(
            6 * self.hidden_size,
            kernel_init=nn.initializers.zeros,
            bias_init=nn.initializers.zeros
        )(nn.silu(c))

        # Each vector: [B, D]
        shift_msa, scale_msa, gate_msa, \
        shift_mlp, scale_mlp, gate_mlp = jnp.split(
            modulation, 6, axis=-1
        )

        # Attention branch
        # AdaLN: h * (1 + scale_msa) + shift_msa
        # scale/shift: [B, D], broadcast across T tokens
        h = modulate(h, shift_msa, scale_msa)  # [B, T, D]

        h = nn.MultiHeadDotProductAttention(
            num_heads=self.num_heads,
            qkv_features=self.hidden_size,
            out_features=self.hidden_size,
            use_bias=True,
            dropout_rate=0.0
        )(h)  # [B, T, D]

        # Gated residual connection; gate_msa: [B, D]
        x = x + gate_msa[:, None, :] * h  # [B, T, D]

        # MLP branch
        h = nn.LayerNorm(
            epsilon=1e-6,
            use_scale=False,
            use_bias=False
        )(x)  # [B, T, D]

        # AdaLN using MLP-specific scale and shift
        h = modulate(h, shift_mlp, scale_mlp)  # [B, T, D]

        # Two-layer feed-forward network
        h = nn.Dense(
            int(self.hidden_size * self.mlp_ratio)
        )(h)  # [B, T, 4D]

        h = nn.gelu(h, approximate=True)  # [B, T, 4D]

        h = nn.Dense(self.hidden_size)(h)  # [B, T, D]

        # Gated MLP residual connection; gate_mlp: [B, D]
        x = x + gate_mlp[:, None, :] * h  # [B, T, D]

        return x  # [B, T, D]

## Final layer

In [ ]:
class FinalLayer(nn.Module):
    """
    Converts final transformer tokens into latent patch predictions
    and optionally produces a separate CLS representation.
    """

    hidden_size: int
    patch_size: int
    out_channels: int
    cls_token_dim: int

    @nn.compact
    def __call__(self, x, c, cls=None):

        # Dimension notation:
        # B = batch size
        # T = number of tokens (patches + optional CLS token)
        # D = transformer hidden dimension
        # P = patch size
        # C = number of latent channels
        # Z = CLS output dimension

        # c: [B, D]
        # Generate adaptive shift and scale parameters.
        # modulation: [B, 2*D]
        modulation = nn.Dense(
            2 * self.hidden_size,
            kernel_init=nn.initializers.zeros,
            bias_init=nn.initializers.zeros
        )(nn.silu(c))

        # shift: [B, D]
        # scale: [B, D]
        shift, scale = jnp.split(
            modulation, 2, axis=-1
        )

        # x: [B, T, D]
        # Normalize transformer representations.
        x = nn.LayerNorm(
            epsilon=1e-6,
            use_scale=False,
            use_bias=False
        )(x)

        # Apply conditioning across all tokens.
        # Output: [B, T, D]
        x = modulate(x, shift, scale)

        if cls is None:

            # Project tokens into flattened latent patches.
            # [B, T, D] -> [B, T, P*P*C]
            x = nn.Dense(
                self.patch_size *
                self.patch_size *
                self.out_channels,
                kernel_init=nn.initializers.zeros
            )(x)

            return x, None

        # Extract CLS token and project it into its target dimension.
        # [B, D] -> [B, Z]
        cls_token = nn.Dense(
            self.cls_token_dim,
            kernel_init=nn.initializers.zeros
        )(x[:, 0])

        # Extract image patch tokens and predict latent patch values.
        # [B, T-1, D] -> [B, T-1, P*P*C]
        patch_tokens = nn.Dense(
            self.patch_size *
            self.patch_size *
            self.out_channels,
            kernel_init=nn.initializers.zeros
        )(x[:, 1:])

        return patch_tokens, cls_token

## Patch emneddings

In [ ]:
def get_1d_sincos_pos_embed(embed_dim, pos):
    """
    Generates 1D sine-cosine positional embeddings.

    Args:
        embed_dim: D, embedding dimension (must be even).
        pos: [M], M positions.

    Returns:
        [M, D], positional embeddings.
    """

    # Generate D/2 frequency indices: [D/2]
    omega = jnp.arange(embed_dim // 2, dtype=jnp.float32)

    # Normalize frequency indices to [0, 2/D, ..., (D-2)/D]
    omega = omega / (embed_dim / 2.0)

    # Convert indices into exponentially decreasing frequencies.
    # omega[d] = 1 / 10000^(2d/D)
    omega = 1.0 / (10000 ** omega)  # [D/2]

    # Outer product: multiply every position by every frequency.
    # [M, 1] * [1, D/2] -> [M, D/2] (each row contains each postiion multiplied by all frequencies)
    out = jnp.einsum("m,d->md", pos.reshape(-1), omega)

    # Encode each position using sine and cosine.
    # Concatenate along feature dimension: [M, D/2] + [M, D/2]
    # Result: [M, D]
    return jnp.concatenate(
        [jnp.sin(out), jnp.cos(out)],
        axis=-1
    )


def get_2d_sincos_pos_embed(embed_dim, grid_size):
    """
    Generates 2D sine-cosine positional embeddings.

    Args:
        embed_dim: D, total embedding dimension (must be divisible by 4).
        grid_size: G, number of patches along each spatial dimension.

    Returns:
        [G*G, D], positional embeddings for all spatial positions.
    """

    # Generate horizontal and vertical coordinate indices: [G]
    grid_h = jnp.arange(grid_size, dtype=jnp.float32)
    grid_w = jnp.arange(grid_size, dtype=jnp.float32)

    # Construct 2D coordinate grids, each of shape [G, G].
    # grid[0] contains horizontal coordinates (columns).
    # grid[1] contains vertical coordinates (rows).
    grid = jnp.meshgrid(grid_w, grid_h, indexing="xy")

    # Encode horizontal coordinates using D/2 features.
    # Input coordinates: [G, G]
    # Flattened positions: [G*G]
    # Output: [G*G, D/2]
    emb_h = get_1d_sincos_pos_embed(
        embed_dim // 2, grid[0]
    )

    # Encode vertical coordinates using the remaining D/2 features.
    # Output: [G*G, D/2]
    emb_w = get_1d_sincos_pos_embed(
        embed_dim // 2, grid[1]
    )

    # Concatenate horizontal and vertical embeddings.
    # [G*G, D/2] + [G*G, D/2] -> [G*G, D]
    return jnp.concatenate([emb_h, emb_w], axis=-1)


class PatchEmbed(nn.Module):
    """
    Converts a spatial latent representation into patch tokens.
    """

    input_size: int
    patch_size: int
    in_channels: int
    hidden_size: int

    @nn.compact
    def __call__(self, x):
        """
        Args:
            x: [B, H, W, C]

        Returns:
            x: [B, N, D]
        """

        # B = batch size
        # H, W = latent spatial dimensions
        # C = input channels
        # P = patch size
        # D = embedding dimension
        # N = number of patches

        B, H, W, C = x.shape

        P = self.patch_size
        D = self.hidden_size

        assert H == self.input_size and W == self.input_size
        assert C == self.in_channels
        assert H % P == 0 and W % P == 0

        # Project each non-overlapping P x P patch into D dimensions.
        # [B, H, W, C] -> [B, H/P, W/P, D]
        x = nn.Conv(
            features=D,
            kernel_size=(P, P),
            strides=(P, P),
            padding="VALID",
            use_bias=True
        )(x)

        # Flatten the spatial grid into a sequence of N patch tokens.
        # [B, H/P, W/P, D] -> [B, N, D]
        x = x.reshape(B, -1, D)

        return x

## Full SiT

In [ ]:
class SiT(nn.Module):
    """
    Diffusion Transformer operating on VAE latent representations.

    Combines patch embeddings, timestep/class conditioning,
    transformer blocks, intermediate representation projectors
    and final latent/CLS prediction heads.
    """

    input_size: int = 32
    patch_size: int = 2
    in_channels: int = 4

    hidden_size: int = 384
    decoder_hidden_size: int = 384

    encoder_depth: int = 8
    depth: int = 12

    num_heads: int = 6
    mlp_ratio: float = 4.0

    num_classes: int = 1000
    class_dropout_prob: float = 0.1

    z_dims: tuple = (768,)
    projector_dim: int = 2048

    qk_norm: bool = False

    def setup(self):

        # Dimension notation:
        # B = batch size
        # H, W = latent spatial dimensions
        # C = latent channels
        # P = patch size
        # T = number of patches + CLS token
        # D = transformer hidden dimension
        # Z = projected representation dimension

        assert self.hidden_size == self.decoder_hidden_size
        assert 0 < self.encoder_depth <= self.depth

        assert self.input_size % self.patch_size == 0
        assert self.hidden_size % self.num_heads == 0

        # Number of spatial patches.
        grid_size = self.input_size // self.patch_size
        num_patches = grid_size * grid_size

        # ------------------------------------------------
        # 1. Patch embedding
        # ------------------------------------------------

        self.x_embedder = PatchEmbed(
            input_size=self.input_size,
            patch_size=self.patch_size,
            in_channels=self.in_channels,
            hidden_size=self.hidden_size
        )

        # ------------------------------------------------
        # 2. Timestep and class embeddings
        # ------------------------------------------------

        self.t_embedder = TimestepEmbedder(
            hidden_size=self.hidden_size
        )

        self.y_embedder = LabelEmbedder(
            num_classes=self.num_classes,
            hidden_size=self.hidden_size,
            dropout_prob=self.class_dropout_prob
        )

        # ------------------------------------------------
        # 3. Fixed positional embedding
        # ------------------------------------------------

        # [1, T, D], including one additional CLS position.
        pos_embed = get_2d_sincos_pos_embed(
            self.hidden_size,
            grid_size
        )

        pos_embed = jnp.concatenate(
            [
                jnp.zeros((1, self.hidden_size)),
                pos_embed
            ],
            axis=0
        )

        self.pos_embed = self.variable(
            "constants",
            "pos_embed",
            lambda: pos_embed[None, :, :]
        )

        # ------------------------------------------------
        # 4. Transformer blocks
        # ------------------------------------------------

        self.blocks = [
            SiTBlock(
                hidden_size=self.hidden_size,
                num_heads=self.num_heads,
                mlp_ratio=self.mlp_ratio,
                qk_norm=self.qk_norm
            )
            for _ in range(self.depth)
        ]

        # ------------------------------------------------
        # 5. Intermediate representation projectors
        # ------------------------------------------------

        '''
        This is the proejctor that projects the intermediate transformer representation into the dimension of the DINO encoded representaiton and correspoding CLS
        '''
        self.projectors = [
            Projector(
                projector_dim=self.projector_dim,
                z_dim=z_dim
            )
            for z_dim in self.z_dims
        ]

        # Original implementation uses the first z_dim
        # as the CLS prediction dimension.
        self.cls_token_dim = self.z_dims[0]

        # ------------------------------------------------
        # 6. External CLS token projection
        # ------------------------------------------------

        self.cls_projectors2 = nn.Dense(
            self.hidden_size
        )

        self.wg_norm = nn.LayerNorm(
            epsilon=1e-6
        )

        # ------------------------------------------------
        # 7. Final prediction layer
        # ------------------------------------------------

        self.final_layer = FinalLayer(
            hidden_size=self.hidden_size,
            patch_size=self.patch_size,
            out_channels=self.in_channels,
            cls_token_dim=self.cls_token_dim
        )

    # ----------------------------------------------------
    # Unpatchify
    # ----------------------------------------------------

    def unpatchify(self, x):

        # x: [B, N, P*P*C]
        #
        # N = number of spatial patches
        # Output: [B, H, W, C] (NHWC)

        B = x.shape[0]

        P = self.patch_size
        C = self.in_channels

        grid_size = self.input_size // P

        x = x.reshape(
            B,
            grid_size,
            grid_size,
            P,
            P,
            C
        )

        # [B, Gh, Gw, P, P, C]
        # -> [B, Gh, P, Gw, P, C]
        x = jnp.transpose(
            x,
            (0, 1, 3, 2, 4, 5)
        )

        # [B, H, W, C]
        x = x.reshape(
            B,
            grid_size * P,
            grid_size * P,
            C
        )

        return x

    # ----------------------------------------------------
    # Forward pass
    # ----------------------------------------------------

        # ----------------------------------------------------
    # Forward pass
    # ----------------------------------------------------

    def __call__(
        self,
        x,
        t,
        y,
        cls_token,
        train=False,
        force_drop_ids=None,
        debug=False
    ):

        def print_shape(name, tensor):
            if debug:
                print(f"{name}: {tensor.shape}")

        # ------------------------------------------------
        # 1. Patch embedding
        # ------------------------------------------------

        print_shape("Input latent", x)

        x = self.x_embedder(x)

        print_shape("Patch embeddings", x)

        # ------------------------------------------------
        # 2. External CLS token
        # ------------------------------------------------

        print_shape("External CLS token", cls_token)

        cls_token = self.cls_projectors2(cls_token)

        print_shape("Projected CLS token", cls_token)

        cls_token = self.wg_norm(cls_token)

        print_shape("Normalized CLS token", cls_token)

        cls_token = cls_token[:, None, :]

        print_shape("CLS token with sequence dimension", cls_token)

        # Prepend CLS token
        x = jnp.concatenate([cls_token, x], axis=1)

        print_shape("Sequence after CLS concatenation", x)

        # Add positional embeddings
        print_shape("Positional embeddings", self.pos_embed.value)

        x = x + self.pos_embed.value

        print_shape("Sequence after positional embeddings", x)

        B, T, D = x.shape

        # ------------------------------------------------
        # 3. Timestep and class conditioning
        # ------------------------------------------------

        print_shape("Input timesteps", t)

        t_embed = self.t_embedder(t)

        print_shape("Timestep embeddings", t_embed)

        print_shape("Input class labels", y)

        y_embed = self.y_embedder(
            y,
            train=train,
            force_drop_ids=force_drop_ids
        )

        print_shape("Class embeddings", y_embed)

        c = t_embed + y_embed

        print_shape("Combined conditioning", c)

        # ------------------------------------------------
        # 4. Transformer blocks
        # ------------------------------------------------

        zs = None

        for i, block in enumerate(self.blocks):

            x = block(x, c)

            if debug:
                print(f"Transformer block {i + 1} output: {x.shape}")

            if (i + 1) == self.encoder_depth:

                zs = [
                    projector(x)
                    for projector in self.projectors
                ]

                if debug:
                    for j, z in enumerate(zs):
                        print(f"Projector {j} output: {z.shape}")

        # ------------------------------------------------
        # 5. Final prediction
        # ------------------------------------------------

        patch_tokens, cls_prediction = self.final_layer(
            x,
            c,
            cls=cls_token
        )

        print_shape("Patch predictions", patch_tokens)
        print_shape("CLS prediction", cls_prediction)

        # ------------------------------------------------
        # 6. Unpatchify
        # ------------------------------------------------

        x = self.unpatchify(patch_tokens)

        print_shape("Reconstructed output latent", x)

        return x, zs, cls_prediction

## Test a single forward pass

In [ ]:
model = SiT(
    input_size=32,
    patch_size=2,
    in_channels=4,

    hidden_size=384,
    decoder_hidden_size=384,

    encoder_depth=8,
    depth=12,

    num_heads=6,
    mlp_ratio=4.0,

    num_classes=1000,
    class_dropout_prob=0.1,

    z_dims=(768,),
    projector_dim=2048,

    qk_norm=False
)


B = 2

# Noisy VAE latent
# [B,H,W,C]
x = jax.random.normal(
    jax.random.PRNGKey(0),
    (B, 32, 32, 4)
)

# Diffusion timestep
# [B]
t = jnp.array(
    [100, 500],
    dtype=jnp.float32
)

# Class labels
# [B]
y = jnp.array(
    [10, 25],
    dtype=jnp.int32
)

# External CLS representation
# [B,Z]
cls_token = jax.random.normal(
    jax.random.PRNGKey(1),
    (B, 768)
)

rng = jax.random.PRNGKey(42)

params_rng, dropout_rng = jax.random.split(rng)

variables = model.init(
    {
        "params": params_rng,
        "dropout": dropout_rng,
    },
    x,
    t,
    y,
    cls_token,
    train=True
)

output, zs, cls_prediction = model.apply(
    variables,
    x,
    t,
    y,
    cls_token,
    train=False,
    debug=True
)

print("\n========== FORWARD PASS RESULTS ==========")

print("Input shape:", x.shape)

print("Output shape:", output.shape)

print("Number of projectors:", len(zs))

for i, z in enumerate(zs):
    print(f"Projector {i} output shape:", z.shape)

print("CLS prediction shape:", cls_prediction.shape)

print("\n========== NUMERICAL CHECKS ==========")

print("Output finite:", jnp.all(jnp.isfinite(output)))
print("Projector finite:", jnp.all(jnp.isfinite(zs[0])))
print("CLS finite:", jnp.all(jnp.isfinite(cls_prediction)))

Input latent: (2, 32, 32, 4)
Patch embeddings: (2, 256, 384)
External CLS token: (2, 768)
Projected CLS token: (2, 384)
Normalized CLS token: (2, 384)
CLS token with sequence dimension: (2, 1, 384)
Sequence after CLS concatenation: (2, 257, 384)
Positional embeddings: (1, 257, 384)
Sequence after positional embeddings: (2, 257, 384)
Input timesteps: (2,)
Timestep embeddings: (2, 384)
Input class labels: (2,)
Class embeddings: (2, 384)
Combined conditioning: (2, 384)
Transformer block 1 output: (2, 257, 384)
Transformer block 2 output: (2, 257, 384)
Transformer block 3 output: (2, 257, 384)
Transformer block 4 output: (2, 257, 384)
Transformer block 5 output: (2, 257, 384)
Transformer block 6 output: (2, 257, 384)
Transformer block 7 output: (2, 257, 384)
Transformer block 8 output: (2, 257, 384)
Projector 0 output: (2, 257, 768)
Transformer block 9 output: (2, 257, 384)
Transformer block 10 output: (2, 257, 384)
Transformer block 11 output: (2, 257, 384)
Transformer block 12 output: (2

In [ ]:
## One training loop
import jax
import jax.numpy as jnp


def mean_flat(x):
    """Mean over all non-batch dimensions. Returns shape [B]."""
    return jnp.mean(
        x,
        axis=tuple(range(1, x.ndim))
    )


class SILoss:

    def __init__(
        self,
        prediction="v",
        path_type="linear",
        weighting="uniform",
        latents_scale=None,
        latents_bias=None,
    ):
        self.prediction = prediction
        self.path_type = path_type
        self.weighting = weighting
        self.latents_scale = latents_scale
        self.latents_bias = latents_bias

    def interpolant(self, t):

        if self.path_type == "linear":

            alpha_t = 1.0 - t
            sigma_t = t

            d_alpha_t = -1.0
            d_sigma_t = 1.0

        elif self.path_type == "cosine":

            alpha_t = jnp.cos(t * jnp.pi / 2.0)
            sigma_t = jnp.sin(t * jnp.pi / 2.0)

            d_alpha_t = -(jnp.pi / 2.0) * jnp.sin(t * jnp.pi / 2.0)
            d_sigma_t = (jnp.pi / 2.0) * jnp.cos(t * jnp.pi / 2.0)

        else:
            raise NotImplementedError(
                f"Unknown path_type: {self.path_type}"
            )

        return alpha_t, sigma_t, d_alpha_t, d_sigma_t

    def __call__(
        self,
        model,
        variables,
        images,
        labels,
        cls_token,
        zs,
        rng,
        model_kwargs=None,
        time_input=None,
        noises=None,
        train=True,
    ):

        B = images.shape[0] # [B, H, W, C]

        model_kwargs = {} if model_kwargs is None else model_kwargs

        # --------------------------------------------------
        # 1. Split random keys
        # --------------------------------------------------

        rng_time, rng_noise, rng_dropout = jax.random.split(rng, 3)

        # --------------------------------------------------
        # 2. Sample timesteps
        # --------------------------------------------------

        if time_input is None:

            if self.weighting == "uniform":

                t = jax.random.uniform(
                    rng_time,
                    shape=(B,),
                    minval=0.0,
                    maxval=1.0,
                )

            elif self.weighting == "lognormal":

                rnd_normal = jax.random.normal(
                    rng_time,
                    shape=(B,),
                )

                sigma = jnp.exp(rnd_normal)

                if self.path_type == "linear":
                    t = sigma / (1.0 + sigma)

                elif self.path_type == "cosine":
                    t = (2.0 / jnp.pi) * jnp.arctan(sigma)

                else:
                    raise NotImplementedError()

            else:
                raise NotImplementedError(
                    f"Unknown weighting: {self.weighting}"
                )

        else:
            t = jnp.asarray(time_input).reshape(B)

        t = t.astype(images.dtype)

        # --------------------------------------------------
        # 3. Sample independent Gaussian noise
        # --------------------------------------------------

        if noises is None:

            rng_x, rng_cls = jax.random.split(rng_noise)

            noise_x = jax.random.normal(
                rng_x,
                shape=images.shape,
                dtype=images.dtype,
            )

            noise_cls = jax.random.normal(
                rng_cls,
                shape=cls_token.shape,
                dtype=cls_token.dtype,
            )

        else:
            noise_x, noise_cls = noises

        # --------------------------------------------------
        # 4. Compute interpolation coefficients
        # --------------------------------------------------

        alpha_t, sigma_t, d_alpha_t, d_sigma_t = self.interpolant(t)

        # Ensure all interpolant quantities have shape [B]
        d_alpha_t = jnp.broadcast_to(
            jnp.asarray(d_alpha_t), t.shape
        )

        d_sigma_t = jnp.broadcast_to(
            jnp.asarray(d_sigma_t), t.shape
        )

        # Expand dimensions for broadcasting
        alpha_x = alpha_t[:, None, None, None]
        sigma_x = sigma_t[:, None, None, None]

        alpha_cls = alpha_t[:, None]
        sigma_cls = sigma_t[:, None]

        d_alpha_x = d_alpha_t[:, None, None, None]
        d_sigma_x = d_sigma_t[:, None, None, None]

        d_alpha_cls = d_alpha_t[:, None]
        d_sigma_cls = d_sigma_t[:, None]
        # --------------------------------------------------
        # 5. Construct noisy inputs
        # --------------------------------------------------

        model_input = (
            alpha_x * images
            + sigma_x * noise_x
        )

        cls_input = (
            alpha_cls * cls_token
            + sigma_cls * noise_cls
        )

        # --------------------------------------------------
        # 6. Construct velocity targets
        # --------------------------------------------------

        if self.prediction == "v":

            d_alpha_x = jnp.asarray(d_alpha_t)[:, None, None, None]
            d_sigma_x = jnp.asarray(d_sigma_t)[:, None, None, None]

            d_alpha_cls = jnp.asarray(d_alpha_t)[:, None]
            d_sigma_cls = jnp.asarray(d_sigma_t)[:, None]

            model_target = (
                d_alpha_x * images
                + d_sigma_x * noise_x
            )

            cls_target = (
                d_alpha_cls * cls_token
                + d_sigma_cls * noise_cls
            )

        else:
            raise NotImplementedError(
                f"Unsupported prediction type: {self.prediction}"
            )

        # --------------------------------------------------
        # 7. Forward pass through SiT
        # --------------------------------------------------

        apply_kwargs = dict(model_kwargs)

        if train:
            model_output, zs_tilde, cls_output = model.apply(
                variables,
                model_input,
                t,
                labels,
                cls_input,
                train=True,
                **apply_kwargs,
                rngs={"dropout": rng_dropout},
            )
        else:
            model_output, zs_tilde, cls_output = model.apply(
                variables,
                model_input,
                t,
                labels,
                cls_input,
                train=False,
                **apply_kwargs,
            )

        # --------------------------------------------------
        # 8. Velocity prediction losses
        # --------------------------------------------------

        denoising_loss = mean_flat(
            (model_output - model_target) ** 2
        )

        denoising_loss_cls = mean_flat(
            (cls_output - cls_target) ** 2
        )

        # --------------------------------------------------
        # 9. DINO projection loss
        # --------------------------------------------------

        proj_loss = 0.0

        assert len(zs) == len(zs_tilde), (
            "Number of teacher feature tensors and projectors differs."
        )

        for z_teacher, z_pred in zip(zs, zs_tilde):

            z_teacher = z_teacher / jnp.maximum(
                jnp.linalg.norm(z_teacher, axis=-1, keepdims=True),
                1e-8,
            )

            z_pred = z_pred / jnp.maximum(
                jnp.linalg.norm(z_pred, axis=-1, keepdims=True),
                1e-8,
            )

            # Negative cosine similarity
            proj_loss += jnp.mean(
                -jnp.sum(z_teacher * z_pred, axis=-1)
            )

        proj_loss = proj_loss / len(zs)

        return (
            denoising_loss,
            proj_loss,
            t,
            (noise_x, noise_cls),
            denoising_loss_cls,
        )

In [ ]:
# --------------------------------------------------
# 2. Create the model
# --------------------------------------------------

B = 2

model = SiT(
    input_size=32,
    patch_size=2,
    in_channels=4,
    hidden_size=384,
    decoder_hidden_size=384,
    encoder_depth=8,
    depth=12,
    num_heads=6,
    mlp_ratio=4.0,
    num_classes=1000,
    class_dropout_prob=0.1,
    z_dims=(768,),
    projector_dim=2048,
    qk_norm=False,
)

# --------------------------------------------------
# 3. Dummy dataset batch
# --------------------------------------------------

rng = jax.random.PRNGKey(0)

rng, rng_x, rng_dino, rng_labels = jax.random.split(rng, 4)

# Clean VAE latents
images = jax.random.normal(
    rng_x, (B, 32, 32, 4)
)

# Dummy DINO features: CLS + 256 patch tokens
dino_features = jax.random.normal(
    rng_dino, (B, 257, 768)
)

# Extract the clean DINO CLS token
cls_token = dino_features[:, 0, :]

# The projection loss expects a list of teacher feature tensors
zs = [dino_features]

# Class labels
labels = jax.random.randint(
    rng_labels, (B,), minval=0, maxval=1000
)

# --------------------------------------------------
# 4. Initialise model parameters
# --------------------------------------------------

rng, rng_init, rng_dropout = jax.random.split(rng, 3)

dummy_t = jnp.ones((B,), dtype=jnp.float32) * 0.5

variables = model.init(
    {"params": rng_init, "dropout": rng_dropout},
    images,
    dummy_t,
    labels,
    cls_token,
    train=True,
)

params = variables["params"]
constants = variables["constants"]

print("Model initialised.")
print("Number of parameter leaves:", len(jax.tree_util.tree_leaves(params)))

# --------------------------------------------------
# 5. Create loss and optimiser
# --------------------------------------------------

loss_fn = SILoss(
    prediction="v",
    path_type="linear",
    weighting="uniform",
)

optimizer = optax.adamw(
    learning_rate=1e-4,
    weight_decay=0.01,
)

opt_state = optimizer.init(params)

# --------------------------------------------------
# 6. Define one training step
# --------------------------------------------------

def train_step(params, constants, opt_state,
               images, labels, cls_token, zs, rng):

    def objective(params):

        variables = {
            "params": params,
            "constants": constants,
        }

        denoising_loss, proj_loss, t, noises, denoising_loss_cls = loss_fn(
            model=model,
            variables=variables,
            images=images,
            labels=labels,
            cls_token=cls_token,
            zs=zs,
            rng=rng,
            train=True,
        )

        loss_x = jnp.mean(denoising_loss)
        loss_cls = jnp.mean(denoising_loss_cls)
        total_loss = loss_x + proj_loss + loss_cls

        metrics = {
            "total_loss": total_loss,
            "denoising_loss": loss_x,
            "projection_loss": proj_loss,
            "cls_loss": loss_cls,
            "mean_t": jnp.mean(t),
        }

        return total_loss, metrics

    (loss, metrics), grads = jax.value_and_grad(
        objective, has_aux=True
    )(params)

    updates, opt_state = optimizer.update(
        grads, opt_state, params
    )

    params = optax.apply_updates(params, updates)

    return params, opt_state, metrics


train_step_jit = jax.jit(train_step)

rng, rng_train = jax.random.split(rng)

params, opt_state, metrics = train_step_jit(
    params,
    constants,
    opt_state,
    images,
    labels,
    cls_token,
    zs,
    rng_train,
)

print("Training step completed!")

for name, value in metrics.items():
    print(f"{name}: {float(value):.6f}")

Model initialised.
Number of parameter leaves: 191
Training step completed!
cls_loss: 2.079759
denoising_loss: 1.969517
mean_t: 0.296945
projection_loss: -0.002379
total_loss: 4.046896
